# 75: Full History Backtest (2009-2026)

**Date:** 2026-01-22  
**Purpose:** Test Check framework on COMPLETE Bitcoin history including bear markets

## Why This Matters:

Previous testing (notebook 74) only used 2023-2026 data:
- **Problem:** 3-year bull market period
- **Missing:** Bear market validation (2018, 2022)
- **Risk:** Strategies might fail in crashes

## Bitcoin Market Cycles to Test:

| Period | Description | Max Drawdown |
|--------|-------------|-------------|
| 2011-2012 | First cycle peak → crash | -93% |
| 2013-2015 | Mt. Gox crash | -86% |
| 2017-2018 | ICO bubble → crash | -83% |
| 2020-2022 | COVID bull → Fed tightening | -76% |
| 2023-2026 | ETF approval bull | ? |

## Key Questions:

1. **Does the framework protect in bear markets?** (2018, 2022 crashes)
2. **Does it capture bull market upside?** (2017, 2020-2021, 2024)
3. **What's the complete cycle performance?** (Full 15+ years)
4. **How many trades over entire history?**
5. **Are results consistent across cycles?**

## Expected Outcome:

If the framework is truly valuable, it should:
- ✅ Reduce drawdowns in 2018 (-83%) and 2022 (-76%)
- ✅ Capture most of bull market gains
- ✅ Show consistent Sharpe ratio improvement
- ⚠️ Might still underperform buy-and-hold in absolute returns

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# VectorBT (mandatory for all backtesting)
try:
    import vectorbt as vbt
    print("✓ VectorBT loaded")
except ImportError:
    print("✗ Installing VectorBT...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "vectorbt"])
    import vectorbt as vbt
    print("✓ VectorBT installed")

# Plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("\n✓ Setup complete!")

In [ ]:
# Configuration
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "brk" / "daily"
GLASSNODE_DIR = PROJECT_ROOT / "data" / "glassnode" / "daily"

# Use FULL history (no train/test split for this analysis)
START_DATE = '2009-01-01'  # Bitcoin genesis
END_DATE = '2026-01-22'    # Today

# Costs
FEES = 0.001
SLIPPAGE = 0.001

print(f"Full history test: {START_DATE} to {END_DATE}")
print(f"Fees: {FEES*100}%, Slippage: {SLIPPAGE*100}%")

## 1. Load Complete Dataset

In [ ]:
def load_metric(name: str, source: str = "brk") -> pd.Series:
    """Load metric as time-indexed Series."""
    if source == "brk":
        path = DATA_DIR / f"{name}.parquet"
    elif source == "glassnode":
        path = GLASSNODE_DIR / f"{name}.parquet"
    else:
        return pd.Series(dtype=float)
    
    if not path.exists():
        return pd.Series(dtype=float)
    
    df = pd.read_parquet(path)
    
    # Normalize
    if 'time' not in df.columns and isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()
        if len(df.columns) == 2:
            df.columns = ['time', 'value']
    
    if 'value' not in df.columns:
        for col in df.columns:
            if col != 'time' and pd.api.types.is_numeric_dtype(df[col]):
                df['value'] = df[col]
                break
    
    if 'time' in df.columns and 'value' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        df = df.set_index('time')['value']
        df = df.sort_index()
        return df
    
    return pd.Series(dtype=float)


# Load all metrics
print("Loading data...")

metrics = {
    'price': ('price', 'brk'),
    'mvrv': ('mvrv', 'brk'),
    'mvrv_sth': ('mvrv_sth', 'brk'),
    'mvrv_lth': ('mvrv_lth', 'brk'),
    'sopr': ('sopr', 'brk'),
    'sopr_sth': ('sopr_sth', 'brk'),
    'sopr_lth': ('sopr_lth', 'brk'),
    'realized_profit': ('realized_profit', 'brk'),
    'realized_loss': ('realized_loss', 'brk'),
    'puell': ('puell_multiple', 'brk'),
    'price_200sma': ('price_200d_sma', 'brk'),
    'funding': ('funding_rate', 'glassnode'),
    'liq_long': ('liquidations_long', 'glassnode'),
    'liq_short': ('liquidations_short', 'glassnode'),
}

df_dict = {}
for name, (metric, source) in metrics.items():
    series = load_metric(metric, source)
    if not series.empty:
        df_dict[name] = series
        print(f"  ✓ {name}: {len(series)} days")
    else:
        print(f"  ✗ {name} (missing)")

df = pd.DataFrame(df_dict)
df = df.fillna(method='ffill')  # Forward fill

print(f"\n✓ Complete dataset: {len(df)} days ({df.index[0].date()} to {df.index[-1].date()})")
print(f"  Time span: {(df.index[-1] - df.index[0]).days / 365.25:.1f} years")

df.head()

## 2. Identify Market Cycles

Segment the data into bull/bear cycles for analysis.

In [ ]:
# Define major market cycles (approximate)
cycles = [
    ('2010-07-01', '2011-11-30', 'Cycle 1: First Bull'),
    ('2011-12-01', '2013-03-31', 'Cycle 1: Bear + Recovery'),
    ('2013-04-01', '2013-11-30', 'Cycle 2: $1000 Bull'),
    ('2013-12-01', '2015-12-31', 'Cycle 2: Mt. Gox Bear'),
    ('2016-01-01', '2017-12-31', 'Cycle 3: ICO Bull'),
    ('2018-01-01', '2019-12-31', 'Cycle 3: 2018 Bear'),
    ('2020-01-01', '2021-11-30', 'Cycle 4: COVID Bull'),
    ('2021-12-01', '2022-12-31', 'Cycle 4: Fed Bear'),
    ('2023-01-01', '2026-01-22', 'Cycle 5: ETF Bull'),
]

# Calculate buy-and-hold returns for each cycle
print("Bitcoin Market Cycles:")
print("="*100)
print(f"{'Period':<35} {'Start Price':>12} {'End Price':>12} {'Return':>12} {'Duration':>10}")
print("-"*100)

for start, end, label in cycles:
    cycle_df = df[(df.index >= start) & (df.index <= end)]
    if len(cycle_df) > 0:
        start_price = cycle_df['price'].iloc[0]
        end_price = cycle_df['price'].iloc[-1]
        returns = (end_price / start_price - 1) * 100
        duration_days = len(cycle_df)
        
        print(f"{label:<35} ${start_price:>11,.0f} ${end_price:>11,.0f} {returns:>11.1f}% {duration_days:>9}d")

print("="*100)

## 3. Generate Signals (Full History)

In [ ]:
def calculate_rolling_zscore(series: pd.Series, window: int) -> pd.Series:
    """Calculate rolling z-score using ONLY past data."""
    mean = series.rolling(window, min_periods=30).mean()
    std = series.rolling(window, min_periods=30).std()
    return (series - mean) / std


def generate_buy_the_dip_entries(df: pd.DataFrame) -> pd.Series:
    """Generate Buy The Dip entry signals."""
    c1 = df['mvrv_sth'] < 1.0
    c2 = df['sopr_sth'] < 1.0
    c3 = (df['realized_profit'] / df['realized_loss']) < 1.0
    c4 = df['funding'] <= 0.0
    c5 = (df['liq_long'] / df['liq_short']) > 1.0
    
    count = c1.astype(int) + c2.astype(int) + c3.astype(int) + c4.astype(int) + c5.astype(int)
    return count >= 4


def generate_lth_distribution_exits(df: pd.DataFrame) -> pd.Series:
    """Generate LTH Distribution exit signals."""
    return (df['mvrv'] > 2.0) & (df['sopr_lth'] > 1.5)


def generate_8metric_exits(df: pd.DataFrame, threshold: int = 6) -> pd.Series:
    """Generate 8-Metric exit signals."""
    mvrv_z = calculate_rolling_zscore(df['mvrv'], 1460)
    mvrv_sth_z = calculate_rolling_zscore(df['mvrv_sth'], 365)
    sopr_z = calculate_rolling_zscore(df['sopr'], 365)
    sopr_sth_z = calculate_rolling_zscore(df['sopr_sth'], 365)
    puell_z = calculate_rolling_zscore(df['puell'], 365)
    mayer_z = calculate_rolling_zscore(df['price'] / df['price_200sma'], 365)
    funding_z = calculate_rolling_zscore(df['funding'], 365)
    
    count = (
        (mvrv_z > 1.5).astype(int) +
        (mvrv_sth_z > 1.25).astype(int) +
        (sopr_z > 1.5).astype(int) +
        (sopr_sth_z > 1.0).astype(int) +
        (mayer_z > 1.0).astype(int) +
        (puell_z > 1.5).astype(int) +
        (funding_z > 1.5).astype(int)
    )
    
    return count >= threshold


# Generate signals
print("Generating signals for full history...\n")

entries = generate_buy_the_dip_entries(df)
lth_exits = generate_lth_distribution_exits(df)
m8_exits = generate_8metric_exits(df, threshold=6)
m4_exits = generate_8metric_exits(df, threshold=4)

print(f"Entry signals (Buy The Dip 4/5): {entries.sum()}")
print(f"LTH Distribution exits: {lth_exits.sum()}")
print(f"8-Metric 6/8 exits: {m8_exits.sum()}")
print(f"8-Metric 4/8 exits: {m4_exits.sum()}")

## 4. VectorBT Backtesting (Full History)

In [ ]:
def backtest_strategy(df: pd.DataFrame, entries: pd.Series, exits: pd.Series, name: str) -> dict:
    """Run backtest using VectorBT."""
    pf = vbt.Portfolio.from_signals(
        close=df['price'],
        entries=entries,
        exits=exits,
        fees=FEES,
        slippage=SLIPPAGE,
        init_cash=10000,
        freq='1D'
    )
    
    return {
        'name': name,
        'portfolio': pf,
        'total_return': pf.total_return() * 100,
        'sharpe': pf.sharpe_ratio(),
        'sortino': pf.sortino_ratio(),
        'calmar': pf.calmar_ratio(),
        'max_dd': pf.max_drawdown() * 100,
        'num_trades': pf.trades.count(),
        'win_rate': pf.trades.win_rate() * 100 if pf.trades.count() > 0 else 0,
    }


# Run backtests
print("\n" + "="*80)
print("FULL HISTORY BACKTEST RESULTS (2009-2026)")
print("="*80)

results = {}
results['lth'] = backtest_strategy(df, entries, lth_exits, "LTH Distribution")
results['8m'] = backtest_strategy(df, entries, m8_exits, "8-Metric 6/8")
results['4m'] = backtest_strategy(df, entries, m4_exits, "8-Metric 4/8")

# Buy and hold
bh_return = (df['price'].iloc[-1] / df['price'].iloc[0] - 1) * 100
bh_equity = (df['price'] / df['price'].iloc[0]) * 10000
bh_dd = (bh_equity / bh_equity.cummax() - 1) * 100

# Results table
print(f"\n{'Strategy':<25} {'Return':>12} {'Sharpe':>8} {'Sortino':>8} {'Max DD':>10} {'Trades':>8} {'Win Rate':>10}")
print("-"*80)

for key, res in results.items():
    print(f"{res['name']:<25} {res['total_return']:>11.1f}% {res['sharpe']:>8.2f} {res['sortino']:>8.2f} {res['max_dd']:>9.1f}% {int(res['num_trades']):>8} {res['win_rate']:>9.1f}%")

print(f"{'Buy & Hold':<25} {bh_return:>11.1f}% {'~1.0':>8} {'~1.0':>8} {bh_dd.min():>9.1f}% {'-':>8} {'-':>10}")
print("="*80)

# Comparison
print("\nVS BUY & HOLD:")
for key, res in results.items():
    diff = res['total_return'] - bh_return
    status = "✓ BEAT" if diff > 0 else "✗ LOST"
    print(f"  {res['name']}: {status} by {abs(diff):.1f}%")

## 5. Equity Curves (Full History)

In [ ]:
# Plot full history equity curves
fig, ax = plt.subplots(figsize=(16, 8))

# Buy & Hold
ax.plot(bh_equity.index, bh_equity.values, label='Buy & Hold', linewidth=2.5, alpha=0.8, color='black', linestyle='--')

# Strategies
for key, res in results.items():
    equity = res['portfolio'].value()
    ax.plot(equity.index, equity.values, label=res['name'], linewidth=2, alpha=0.8)

# Mark major events
events = [
    ('2013-12-01', 'Mt. Gox'),
    ('2017-12-17', '2017 Peak'),
    ('2018-12-15', '2018 Bottom'),
    ('2021-11-10', '2021 Peak'),
    ('2022-11-21', '2022 Bottom'),
]

for date, label in events:
    if pd.Timestamp(date) in bh_equity.index:
        ax.axvline(pd.Timestamp(date), color='gray', linestyle=':', alpha=0.5, linewidth=1)
        ax.text(pd.Timestamp(date), ax.get_ylim()[1]*0.9, label, rotation=90, fontsize=8, alpha=0.7)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Portfolio Value ($)', fontsize=12)
ax.set_title('Check Framework vs Buy & Hold: Complete Bitcoin History (2009-2026)', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal Portfolio Values (started with $10,000):")
print(f"  Buy & Hold: ${bh_equity.iloc[-1]:,.0f}")
for key, res in results.items():
    final = res['portfolio'].value().iloc[-1]
    print(f"  {res['name']}: ${final:,.0f}")

## 6. Drawdown Comparison (Critical!)

In [ ]:
# Drawdown plots
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

# Buy & Hold
axes[0].fill_between(bh_dd.index, bh_dd.values, 0, alpha=0.3, color='black')
axes[0].plot(bh_dd.index, bh_dd.values, color='black', linewidth=1.5)
axes[0].set_title(f'Buy & Hold\nMax DD: {bh_dd.min():.1f}%', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Drawdown (%)')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(-50, color='red', linestyle='--', alpha=0.5, label='-50% line')
axes[0].legend()

# Strategies
for idx, (key, res) in enumerate(results.items(), 1):
    dd = res['portfolio'].drawdown() * 100
    axes[idx].fill_between(dd.index, dd.values, 0, alpha=0.3)
    axes[idx].plot(dd.index, dd.values, linewidth=1.5)
    axes[idx].set_title(f"{res['name']}\nMax DD: {res['max_dd']:.1f}%", fontweight='bold', fontsize=12)
    axes[idx].set_ylabel('Drawdown (%)')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].axhline(-50, color='red', linestyle='--', alpha=0.5, label='-50% line')
    axes[idx].legend()

plt.tight_layout()
plt.show()

print("\nDrawdown Protection:")
print(f"  Buy & Hold: {bh_dd.min():.1f}%")
for key, res in results.items():
    protection = bh_dd.min() - res['max_dd']
    print(f"  {res['name']}: {res['max_dd']:.1f}% (protected {abs(protection):.1f}%)")

## 7. Cycle-by-Cycle Performance

In [ ]:
# Analyze performance in each major cycle
print("\nCycle-by-Cycle Performance:")
print("="*120)

key_cycles = [
    ('2017-01-01', '2017-12-31', '2017 Bull'),
    ('2018-01-01', '2018-12-31', '2018 Bear'),
    ('2020-01-01', '2021-11-30', '2020-2021 Bull'),
    ('2021-12-01', '2022-12-31', '2022 Bear'),
    ('2023-01-01', '2026-01-22', '2023-2026 Bull'),
]

cycle_results = []

for start, end, label in key_cycles:
    cycle_df = df[(df.index >= start) & (df.index <= end)].copy()
    
    if len(cycle_df) < 30:
        continue
    
    # Buy and hold for this cycle
    bh_ret = (cycle_df['price'].iloc[-1] / cycle_df['price'].iloc[0] - 1) * 100
    
    print(f"\n{label} ({start} to {end}):")
    print(f"  Buy & Hold: {bh_ret:+.1f}%")
    
    # Test each strategy on this cycle
    cycle_entries = generate_buy_the_dip_entries(cycle_df)
    cycle_lth_exits = generate_lth_distribution_exits(cycle_df)
    cycle_8m_exits = generate_8metric_exits(cycle_df, 6)
    
    for name, exits in [('LTH Distribution', cycle_lth_exits), ('8-Metric 6/8', cycle_8m_exits)]:
        try:
            pf = vbt.Portfolio.from_signals(
                close=cycle_df['price'],
                entries=cycle_entries,
                exits=exits,
                fees=FEES,
                slippage=SLIPPAGE,
                init_cash=10000,
                freq='1D'
            )
            ret = pf.total_return() * 100
            diff = ret - bh_ret
            status = "✓" if diff > 0 else "✗"
            print(f"  {name}: {ret:+.1f}% ({status} {diff:+.1f}% vs B&H)")
        except:
            print(f"  {name}: No trades")

print("\n" + "="*120)

## 8. Bear Market Protection Analysis

Critical test: Did the framework protect capital in 2018 and 2022?

In [ ]:
# Focus on 2018 and 2022 bear markets
bear_markets = [
    ('2017-12-17', '2018-12-15', '2018 Bear Market'),
    ('2021-11-10', '2022-11-21', '2022 Bear Market'),
]

print("\n" + "="*80)
print("BEAR MARKET PROTECTION TEST")
print("="*80)

for start, end, label in bear_markets:
    bear_df = df[(df.index >= start) & (df.index <= end)].copy()
    
    if len(bear_df) == 0:
        continue
    
    bh_ret = (bear_df['price'].iloc[-1] / bear_df['price'].iloc[0] - 1) * 100
    
    print(f"\n{label}:")
    print(f"  Duration: {len(bear_df)} days ({len(bear_df)/365.25:.1f} years)")
    print(f"  Buy & Hold: {bh_ret:.1f}%")
    
    # Test strategies
    bear_entries = generate_buy_the_dip_entries(bear_df)
    bear_lth_exits = generate_lth_distribution_exits(bear_df)
    bear_8m_exits = generate_8metric_exits(bear_df, 6)
    
    for name, exits in [('LTH Distribution', bear_lth_exits), ('8-Metric 6/8', bear_8m_exits)]:
        try:
            pf = vbt.Portfolio.from_signals(
                close=bear_df['price'],
                entries=bear_entries,
                exits=exits,
                fees=FEES,
                slippage=SLIPPAGE,
                init_cash=10000,
                freq='1D'
            )
            ret = pf.total_return() * 100
            protection = bh_ret - ret
            print(f"  {name}: {ret:.1f}% (protected {abs(protection):.1f}% from losses)")
        except:
            print(f"  {name}: No trades (stayed in cash!)")

print("\n" + "="*80)

## 9. Full History Conclusion

In [ ]:
print("="*80)
print("FULL HISTORY VERDICT (2009-2026)")
print("="*80)

print(f"\n1. ABSOLUTE RETURNS ({(df.index[-1] - df.index[0]).days / 365.25:.1f} years):")
print(f"   Buy & Hold: {bh_return:+.1f}%")
for key, res in results.items():
    diff = res['total_return'] - bh_return
    print(f"   {res['name']}: {res['total_return']:+.1f}% ({diff:+.1f}%)")

print(f"\n2. RISK MANAGEMENT:")
print(f"   Buy & Hold Max DD: {bh_dd.min():.1f}%")
for key, res in results.items():
    protection = bh_dd.min() - res['max_dd']
    print(f"   {res['name']}: {res['max_dd']:.1f}% (protected {abs(protection):.1f}%)")

print(f"\n3. RISK-ADJUSTED:")
for key, res in results.items():
    print(f"   {res['name']}: Sharpe {res['sharpe']:.2f}, Sortino {res['sortino']:.2f}, Calmar {res['calmar']:.2f}")

print(f"\n4. TRADING ACTIVITY:")
for key, res in results.items():
    years = (df.index[-1] - df.index[0]).days / 365.25
    trades_per_year = res['num_trades'] / years
    print(f"   {res['name']}: {int(res['num_trades'])} trades ({trades_per_year:.1f}/year), {res['win_rate']:.1f}% win rate")

print("\n5. KEY INSIGHTS:")
best = max(results.values(), key=lambda x: x['sharpe'])
print(f"   • Best risk-adjusted: {best['name']} (Sharpe {best['sharpe']:.2f})")

best_abs = max(results.values(), key=lambda x: x['total_return'])
if best_abs['total_return'] > bh_return:
    print(f"   • ✅ {best_abs['name']} BEATS buy-and-hold by {best_abs['total_return'] - bh_return:.1f}%!")
else:
    print(f"   • ✗ All strategies underperform buy-and-hold in absolute returns")

best_dd = min(results.values(), key=lambda x: x['max_dd'])
dd_protection = bh_dd.min() - best_dd['max_dd']
print(f"   • Best drawdown protection: {best_dd['name']} (-{best_dd['max_dd']:.1f}% vs {bh_dd.min():.1f}%)")

print("\n" + "="*80)

## Next Steps:

Based on full history results:
1. If framework beats B&H → Move to position sizing (notebook 76)
2. If framework provides good DD protection → Consider hybrid approach
3. If framework fails → Re-evaluate entire approach